In [0]:
USE CATALOG projectcatalog

In [0]:
use schema gold_schema

In [0]:
-- 2. Check Silver source
SELECT * FROM sliverschemasales.tblprjproductsilver LIMIT 20;

product_id,product_name,category,brand,price,price_category,silver_load_created_date,source_file,data_layer
P0001,Clearly Its,Beauty,Nike,1868.54,HIGH,2026-08-11T18:09:53.045Z,ProductsFile.parquet,SILVER
P0002,Production Clear,Beauty,Apple,587.13,MEDIUM,2026-08-11T18:09:53.045Z,ProductsFile.parquet,SILVER
P0003,Culture Coach,Home,Revlon,1599.24,HIGH,2026-08-11T18:09:53.045Z,ProductsFile.parquet,SILVER
P0004,Movement Part,Sports,Lg,651.71,MEDIUM,2026-08-11T18:09:53.045Z,ProductsFile.parquet,SILVER
P0005,Fact Name,Clothing,Samsung,1861.78,HIGH,2026-08-11T18:09:53.045Z,ProductsFile.parquet,SILVER
P0006,Usually Stop,Toys,Adidas,936.36,MEDIUM,2026-08-11T18:09:53.045Z,ProductsFile.parquet,SILVER
P0007,Reveal Current,Sports,Adidas,1954.02,HIGH,2026-08-11T18:09:53.045Z,ProductsFile.parquet,SILVER
P0008,Force Language,Beauty,Puma,1251.26,MEDIUM,2026-08-11T18:09:53.045Z,ProductsFile.parquet,SILVER
P0009,Stage Leg,Clothing,Samsung,1247.15,MEDIUM,2026-08-11T18:09:53.045Z,ProductsFile.parquet,SILVER
P0010,Leader Then,Sports,Sony,975.53,MEDIUM,2026-08-11T18:09:53.045Z,ProductsFile.parquet,SILVER


In [0]:
-- 3. Create Gold product dimension
CREATE TABLE IF NOT EXISTS dim_product
(
    product_key            BIGINT,
    product_id             STRING,
    product_name           STRING,
    category               STRING,
    brand                  STRING,
    price                  DECIMAL(18,2),
    price_category         STRING,
    source_file            STRING,
    silver_load_datetime   TIMESTAMP,
    gold_created_datetime  TIMESTAMP,
    gold_updated_datetime  TIMESTAMP
)
USING DELTA;
 


In [0]:
-- 4. Prepare latest source record for every product
CREATE OR REPLACE TEMP VIEW products_gold_source AS
SELECT
    XXHASH64(product_id) AS product_key,
    product_id,
    product_name,
    category,
    brand,
    CAST(price AS DECIMAL(18,2)) AS price,
    price_category,
    source_file,
    silver_load_created_date AS silver_load_datetime,
    CURRENT_TIMESTAMP() AS load_datetime
FROM
(
    SELECT
        *,
        ROW_NUMBER() OVER
        (
            PARTITION BY product_id
            ORDER BY silver_load_created_date DESC
        ) AS row_num

    FROM sliverschemasales.tblprjproductsilver
) source 
WHERE row_num = 1;


In [0]:
-- 5. SCD Type 1 MERGE
MERGE INTO dim_product AS target 
USING products_gold_source AS source
ON target.product_id = source.product_id
WHEN MATCHED AND
(
       NOT (target.product_name   <=> source.product_name)
    OR NOT (target.category       <=> source.category)
    OR NOT (target.brand          <=> source.brand)
    OR NOT (target.price          <=> source.price)
    OR NOT (target.price_category <=> source.price_category)
    OR NOT (target.source_file    <=> source.source_file)
)
THEN UPDATE SET
    target.product_name = source.product_name,
    target.category = source.category,
    target.brand = source.brand,
    target.price = source.price,
    target.price_category = source.price_category,
    target.source_file = source.source_file,
    target.silver_load_datetime = source.silver_load_datetime,
    target.gold_updated_datetime = source.load_datetime
WHEN NOT MATCHED
THEN INSERT
(
    product_key,
    product_id,
    product_name,
    category,
    brand,
    price,
    price_category,
    source_file,
    silver_load_datetime,
    gold_created_datetime,
    gold_updated_datetime
)
VALUES
(
    source.product_key,
    source.product_id,
    source.product_name,
    source.category,
    source.brand,
    source.price,
    source.price_category,
    source.source_file,
    source.silver_load_datetime,
    source.load_datetime,
    source.load_datetime
);



num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
490,0,0,490


In [0]:
-- 6. Preview Gold product dimension
SELECT * FROM  dim_product ORDER BY product_id;


product_key,product_id,product_name,category,brand,price,price_category,source_file,silver_load_datetime,gold_created_datetime,gold_updated_datetime
-7303578145898241718,P0001,Clearly Its,Beauty,Nike,1868.54,HIGH,ProductsFile.parquet,2026-08-11T18:09:53.045Z,2026-09-08T10:07:31.853Z,2026-09-08T10:07:31.853Z
-8083715307953110988,P0002,Production Clear,Beauty,Apple,587.13,MEDIUM,ProductsFile.parquet,2026-08-11T18:09:53.045Z,2026-09-08T10:07:31.853Z,2026-09-08T10:07:31.853Z
-4744954237822947751,P0003,Culture Coach,Home,Revlon,1599.24,HIGH,ProductsFile.parquet,2026-08-11T18:09:53.045Z,2026-09-08T10:07:31.853Z,2026-09-08T10:07:31.853Z
7885827818829170315,P0004,Movement Part,Sports,Lg,651.71,MEDIUM,ProductsFile.parquet,2026-08-11T18:09:53.045Z,2026-09-08T10:07:31.853Z,2026-09-08T10:07:31.853Z
4118278677759797376,P0005,Fact Name,Clothing,Samsung,1861.78,HIGH,ProductsFile.parquet,2026-08-11T18:09:53.045Z,2026-09-08T10:07:31.853Z,2026-09-08T10:07:31.853Z
-1864589561315923401,P0006,Usually Stop,Toys,Adidas,936.36,MEDIUM,ProductsFile.parquet,2026-08-11T18:09:53.045Z,2026-09-08T10:07:31.853Z,2026-09-08T10:07:31.853Z
8051383078171288891,P0007,Reveal Current,Sports,Adidas,1954.02,HIGH,ProductsFile.parquet,2026-08-11T18:09:53.045Z,2026-09-08T10:07:31.853Z,2026-09-08T10:07:31.853Z
4620441700520709352,P0008,Force Language,Beauty,Puma,1251.26,MEDIUM,ProductsFile.parquet,2026-08-11T18:09:53.045Z,2026-09-08T10:07:31.853Z,2026-09-08T10:07:31.853Z
-8983413223716387440,P0009,Stage Leg,Clothing,Samsung,1247.15,MEDIUM,ProductsFile.parquet,2026-08-11T18:09:53.045Z,2026-09-08T10:07:31.853Z,2026-09-08T10:07:31.853Z
1590853133898342123,P0010,Leader Then,Sports,Sony,975.53,MEDIUM,ProductsFile.parquet,2026-08-11T18:09:53.045Z,2026-09-08T10:07:31.853Z,2026-09-08T10:07:31.853Z


In [0]:
-- 7. Duplicate product check 
SELECT product_id, COUNT(*) AS duplicate_count FROM dim_product GROUP BY product_id HAVING COUNT(*) > 1;

product_id,duplicate_count


In [0]:
-- 8. Silver-to-Gold count reconciliation
SELECT
    (
        SELECT COUNT(DISTINCT product_id)
        FROM sliverschemasales.tblprjproductsilver
    ) AS silver_product_count,
    (
        SELECT COUNT(*)
        FROM dim_product
    ) AS gold_product_count;

silver_product_count,gold_product_count
490,490


In [0]:
-- 9. Product summary by category
CREATE OR REPLACE VIEW vw_product_category_summary
AS
SELECT
    category,
    COUNT(*) AS total_products,
    ROUND(
        AVG(price),
        2
    ) AS average_price,
    MIN(price) AS minimum_price,
    MAX(price) AS maximum_price FROM dim_product
GROUP BY category;


In [0]:
SELECT * FROM vw_product_category_summary ORDER BY total_products DESC;


category,total_products,average_price,minimum_price,maximum_price
Electronics,99,1095.52,12.35,1983.25
Beauty,93,1069.76,20.68,1935.07
Toys,84,993.15,38.43,1971.77
Sports,76,1062.94,48.00,1967.84
Clothing,72,1022.73,15.91,1928.38
Home,66,978.22,22.30,1876.88


In [0]:
-- 10. Product summary by brand
CREATE OR REPLACE VIEW   vw_product_brand_summary
AS
SELECT
    brand,
    COUNT(*) AS total_products,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(SUM(price),2) AS total_catalog_value
FROM dim_product
GROUP BY brand;

In [0]:
SELECT * FROM vw_product_brand_summary

brand,total_products,average_price,total_catalog_value
Nike,48,1044.10,50116.67
Apple,46,1148.75,52842.28
Revlon,41,1070.40,43886.23
Lg,41,994.91,40791.37
Samsung,59,923.02,54458.12
Adidas,60,1156.38,69382.50
Puma,47,1050.51,49373.96
Sony,49,1021.43,50050.29
Dell,45,978.25,44021.15
Lenovo,54,1026.44,55427.91


In [0]:
-- 11. Product summary by price category
CREATE OR REPLACE VIEW
    vw_product_price_summary
AS
SELECT
    price_category,
    COUNT(*) AS total_products,
    ROUND(AVG(price),2) AS average_price,
    MIN(price) AS minimum_price,
    MAX(price) AS maximum_price
FROM dim_product
GROUP BY price_category;

In [0]:
SELECT  * FROM vw_product_price_summary

price_category,total_products,average_price,minimum_price,maximum_price
HIGH,129,1733.35,1500.97,1983.25
MEDIUM,254,1025.47,503.61,1494.78
LOW,107,245.59,12.35,492.92


In [0]:

SELECT * FROM vw_product_price_summary
ORDER BY
    CASE price_category
        WHEN 'LOW' THEN 1
        WHEN 'MEDIUM' THEN 2
        WHEN 'HIGH' THEN 3
        ELSE 4
    END;

price_category,total_products,average_price,minimum_price,maximum_price
LOW,107,245.59,12.35,492.92
MEDIUM,254,1025.47,503.61,1494.78
HIGH,129,1733.35,1500.97,1983.25


In [0]:
-- 12. Identify products updated after initial insert
SELECT * FROM dim_product WHERE gold_updated_datetime > gold_created_datetime ORDER BY gold_updated_datetime DESC;


product_key,product_id,product_name,category,brand,price,price_category,source_file,silver_load_datetime,gold_created_datetime,gold_updated_datetime


In [0]:
-- 13. Optional optimization
OPTIMIZE dim_product ZORDER BY(product_id,category,brand);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 13793), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1788862618884, 1788862619949, 8, 0, null, List(0, 0), null, 11, 11, 0, 0, null, null, 0)"
